In [4]:
# ==============================================================================
# 2026改修版: kNN + 外れ値解析用フル版（旧特徴量ロジック準拠）
# ==============================================================================

suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
  library(Metrics)
  library(stringr)
})

set.seed(42)

# -------------------------------
# (F)(G) 設定：パスと対象ファイル
# -------------------------------
base_path   <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
file_names  <- c("m_all_FP_rebuilt.csv")

# ★4目的変数を旧コードと同じように定義
target_vars_all <- c("Jsc", "Voc", "FF", "PCEmax")
# このスクリプトで実際に学習するターゲット
target_vars_run <- c("FF")

# 出力先設定（outlier 配下）
today    <- format(Sys.Date(), "%Y%m%d")
out_root <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/after_20251225/outlier"
model_name <- "kNN_oldFeatLogic"

# (C)(D)(E) 除外変数設定
inappropriate_vars <- c(
  "Jsccal", "JscJsccal", "PCEaTA", "PCEaTAFinal", "EQEABEST", "Rshthin", "Dresistance",
  "mobilityeblendOFET", "mobilityep1MOFET", "mobilityhblendSCLC", "mobilityhnMSCLC",
  "mobilityhp1MSCLC", "IonIoffF", "DRMS", "Var.1"
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

# -------------------------------
# HELPERS
# -------------------------------
safe_r2 <- function(y, p) {
  r <- suppressWarnings(cor(y, p))
  if (is.na(r)) return(NA_real_)
  as.numeric(r^2)
}

calc_knn_importance <- function(model, data_scaled, target, features) {
  base_pred <- predict(model, data_scaled[, features, drop = FALSE])
  base_r2   <- safe_r2(data_scaled[[target]], base_pred)
  
  out_imps <- list()
  for (col in features) {
    temp <- data_scaled
    temp[[col]] <- sample(temp[[col]])
    shuffled_pred <- tryCatch(
      predict(model, temp[, features, drop = FALSE]),
      error = function(e) NA
    )
    if (all(is.na(shuffled_pred))) {
      out_imps[[col]] <- 0
    } else {
      new_r2 <- safe_r2(data_scaled[[target]], shuffled_pred)
      out_imps[[col]] <- max(0, base_r2 - new_r2)
    }
  }
  data.frame(Feature = names(out_imps), Importance = as.numeric(unlist(out_imps)))
}

flag_high_error <- function(df_pred, quantile_prob = 0.9) {
  thr <- stats::quantile(df_pred$AbsError, quantile_prob, na.rm = TRUE)
  df_pred$HighErrorFlag      <- df_pred$AbsError >= thr
  df_pred$HighErrorThreshold <- thr
  df_pred
}

# -------------------------------
# MAIN LOOP
# -------------------------------
summary_list    <- list()
importance_list <- list()

for (file in file_names) {
  filepath <- file.path(base_path, file)
  if (!file.exists(filepath)) {
    cat("[WARN] File not found:", filepath, "\n")
    next
  }
  
  df_raw <- tryCatch(
    read.csv(filepath, row.names = 1, check.names = FALSE),
    error = function(e) NULL
  )
  if (is.null(df_raw)) {
    cat("[WARN] Failed to read:", filepath, "\n")
    next
  }
  if ("X" %in% colnames(df_raw)) df_raw$X <- NULL
  
  for (target in target_vars_run) {
    if (!target %in% colnames(df_raw)) {
      cat("[WARN] Target not found:", target, "in", file, "\n")
      next
    }
    
    # 出力ディレクトリ
    tag     <- paste0(tools::file_path_sans_ext(file), "_", target)
    run_dir <- file.path(out_root, paste0(model_name, "_", tag))
    if (!dir.exists(run_dir)) dir.create(run_dir, recursive = TRUE)
    
    # ----------------------------
    # クレンジング
    # ----------------------------
    df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
    if (nrow(df_temp) < 20) {
      cat("[WARN] Too few rows after na.omit:", nrow(df_temp), "for", tag, "\n")
      next
    }
    
    # ----------------------------
    # (C)(D)(E) 変数除外（★旧ロジック）
    # ----------------------------
    # 4 目的変数をすべて除外し、それ以外の数値列を候補に
    features <- setdiff(colnames(df_temp), target_vars_all)
    features <- setdiff(features, inappropriate_vars)
    if (length(physical_exclude_patterns) > 0) {
      features <- features[!grepl(paste(physical_exclude_patterns, collapse = "|"), features)]
    }
    
    # ゼロ分散除外
    vars     <- sapply(df_temp[, features, drop = FALSE], var)
    features <- names(vars)[vars > 0 & !is.na(vars)]
    
    # (H) 多重共線性対策
    if (length(features) > 1) {
      cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
      cor_mat[is.na(cor_mat)] <- 0
      high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
      if (length(high_corr) > 0) features <- features[-high_corr]
    }
    if (length(features) == 0) {
      cat("[WARN] No usable features for", tag, "\n")
      next
    }
    
    # ----------------------------
    # (I) スケーリング [-1, 1]
    # ----------------------------
    pp <- preProcess(df_temp[, features, drop = FALSE], method = "range")
    df_scaled <- df_temp
    df_scaled[, features] <- predict(pp, df_temp[, features]) * 2 - 1
    df_scaled$SampleID <- rownames(df_scaled)
    
    # ----------------------------
    # (K) CV10設定
    # ----------------------------
    train_ctrl <- trainControl(
      method = "cv",
      number = 10,
      savePredictions = "final",
      returnResamp   = "all"
    )
    tune_grid <- expand.grid(k = seq(1, 21, by = 2))
    
    # 学習
    model <- tryCatch(
      train(
        x = df_scaled[, features, drop = FALSE],
        y = df_scaled[[target]],
        method      = "knn",
        trControl   = train_ctrl,
        tuneGrid    = tune_grid,
        metric      = "RMSE"
      ),
      error = function(e) {
        cat("  [ERROR] kNN failed:", e$message, "\n")
        return(NULL)
      }
    )
    if (is.null(model)) next
    
    best_k   <- model$bestTune$k
    row_best <- model$results[model$results$k == best_k, ]
    
    cat(sprintf(
      "[BEST] %s - %s | k=%d | RMSE=%.4f, Rsq=%.4f\n",
      file, target, best_k,
      row_best$RMSE, row_best$Rsquared
    ))
    
    # チューニング結果保存
    tune_tbl <- model$results
    tune_tbl$File   <- file
    tune_tbl$Target <- target
    tune_tbl$Model  <- model_name
    write.csv(
      tune_tbl,
      file.path(run_dir, paste0(tag, "_tuning_results.csv")),
      row.names = FALSE
    )
    
    best_row <- row_best
    best_row$File   <- file
    best_row$Target <- target
    best_row$Model  <- model_name
    write.csv(
      best_row,
      file.path(run_dir, paste0(tag, "_best_params.csv")),
      row.names = FALSE
    )
    
    # ----------------------------
    # OOF 予測 + foldID + 誤差
    # ----------------------------
    pred_oof <- model$pred %>%
      dplyr::filter(k == model$bestTune$k) %>%
      mutate(
        SampleID = rownames(df_scaled)[rowIndex],
        foldID   = Resample,
        Type     = "CV10_OOF",
        Observed = obs,
        Predicted = pred,
        Error    = Predicted - Observed,
        AbsError = abs(Error)
      ) %>%
      dplyr::select(SampleID, foldID, Observed, Predicted, Error, AbsError, Type)
    
    pred_oof <- flag_high_error(pred_oof, quantile_prob = 0.9)
    
    # 特徴量テーブルとの結合
    feat_df <- df_scaled[, c("SampleID", features), drop = FALSE]
    pred_with_feat <- pred_oof %>% left_join(feat_df, by = "SampleID")
    
    # モデル保存
    saveRDS(model, file.path(run_dir, paste0(tag, "_model.rds")))
    saveRDS(pp,    file.path(run_dir, paste0(tag, "_preprocess.rds")))
    
    # CSV 出力
    write.csv(
      pred_oof,
      file.path(run_dir, paste0(tag, "_pred_OOF_withError.csv")),
      row.names = FALSE
    )
    write.csv(
      pred_with_feat,
      file.path(run_dir, paste0(tag, "_pred_OOF_withError_andFeatures.csv")),
      row.names = FALSE
    )
    write.csv(
      pred_with_feat[pred_with_feat$HighErrorFlag, ],
      file.path(run_dir, paste0(tag, "_highErrorSamples.csv")),
      row.names = FALSE
    )
    
    # 重要度
    imp_df <- calc_knn_importance(model, df_scaled, target, features)
    imp_df$File   <- file
    imp_df$Target <- target
    imp_df$Model  <- model_name
    importance_list[[length(importance_list) + 1]] <- imp_df
    
    # サマリー集計
    cv10_r2   <- safe_r2(pred_oof$Observed, pred_oof$Predicted)
    cv10_rmse <- rmse(pred_oof$Observed, pred_oof$Predicted)
    
    summary_list[[length(summary_list) + 1]] <- data.frame(
      File       = file,
      Target     = target,
      Model      = model_name,
      CV10_R2    = cv10_r2,
      CV10_RMSE  = cv10_rmse,
      n_samples  = nrow(df_scaled),
      n_features = length(features),
      Best_K     = best_k,
      HighErrorThreshold = unique(pred_oof$HighErrorThreshold)
    )
    
    cat(sprintf("  Finished: %s - %s | CV10_R2: %.3f | n=%d | p=%d\n",
                file, target, cv10_r2, nrow(df_scaled), length(features)))
  }
}

# -------------------------------
# グローバル CSV 出力
# -------------------------------
if (length(summary_list) > 0) {
  all_summary <- do.call(rbind, summary_list)
  write.csv(
    all_summary,
    file.path(out_root, paste0("all_summary_CV10_", model_name, ".csv")),
    row.names = FALSE
  )
}

if (length(importance_list) > 0) {
  all_imp <- do.call(rbind, importance_list)
  write.csv(
    all_imp,
    file.path(out_root, paste0("all_importance_", model_name, ".csv")),
    row.names = FALSE
  )
}

cat("\n[DONE] kNN Outlier-aware Analysis (old feature-selection logic) completed.\n")


[BEST] m_all_FP_rebuilt.csv - FF | k=3 | RMSE=0.0791, Rsq=0.5840
  Finished: m_all_FP_rebuilt.csv - FF | CV10_R2: 0.584 | n=1045 | p=283

[DONE] kNN Outlier-aware Analysis (old feature-selection logic) completed.


In [5]:
suppressPackageStartupMessages({
  library(caret)
  library(dplyr)
})

# 設定
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"
file_names <- c(
  "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
  "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
  "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
  "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
)

target_vars_all <- c("Jsc", "Voc", "FF", "PCEmax")
inappropriate_vars <- c(
  "Jsccal", "JscJsccal", "PCEaTA", "PCEaTAFinal", "EQEABEST", "Rshthin", "Dresistance",
  "mobilityeblendOFET", "mobilityep1MOFET", "mobilityhblendSCLC", "mobilityhnMSCLC",
  "mobilityhp1MSCLC", "IonIoffF", "DRMS", "Var.1"
)
physical_exclude_patterns <- c("X2DGIWAXD", "X2DGIXD", "widthfibril")

cat(sprintf("%-25s | %-12s | %-12s\n", "Dataset", "Samples(B->A)", "Features(B->A)"))
cat(paste0(rep("-", 55), collapse=""), "\n")

for (file in file_names) {
  df_raw <- tryCatch(read.csv(file.path(base_path, file), row.names = 1, check.names = FALSE), error = function(e) NULL)
  if (is.null(df_raw)) next
  
  # Before stats (1列目のIDは除外)
  n_before <- nrow(df_raw)
  p_before <- ncol(df_raw) - length(intersect(colnames(df_raw), target_vars_all))
  
  # Processing
  df_temp <- df_raw[, sapply(df_raw, is.numeric)] %>% na.omit()
  features <- setdiff(colnames(df_temp), target_vars_all)
  features <- setdiff(features, inappropriate_vars)
  if (length(physical_exclude_patterns) > 0) {
    features <- features[!grepl(paste(physical_exclude_patterns, collapse = "|"), features)]
  }
  
  # SD=0 removal
  vars <- sapply(df_temp[, features, drop = FALSE], var)
  features <- names(vars)[vars > 0 & !is.na(vars)]
  
  # Multicollinearity (0.99999)
  if (length(features) > 1) {
    cor_mat <- cor(df_temp[, features, drop = FALSE], use = "pairwise.complete.obs")
    cor_mat[is.na(cor_mat)] <- 0
    high_corr <- findCorrelation(cor_mat, cutoff = 0.99999)
    if (length(high_corr) > 0) features <- features[-high_corr]
  }
  
  # After stats
  n_after <- nrow(df_temp)
  p_after <- length(features)
  
  cat(sprintf("%-25s | %4d -> %4d | %4d -> %4d\n", file, n_before, n_after, p_before, p_after))
}

Dataset                   | Samples(B->A) | Features(B->A)
------------------------------------------------------- 
n_base.csv                |  358 ->  358 |    7 ->    7
n_base_OH_rebuilt.csv     |  358 ->  358 |  319 ->  137
n_base_FP_rebuilt.csv     |  358 ->  358 |  213 ->   91
n_all.csv                 |   72 ->   72 |   33 ->   24
n_all_OH_rebuilt.csv      |   72 ->   72 |  345 ->   48
n_all_FP_rebuilt.csv      |   72 ->   72 |  239 ->   44
m_base.csv                | 1045 -> 1045 |    7 ->    7
m_base_OH_rebuilt.csv     | 1045 -> 1045 |  319 ->  318
m_base_FP_rebuilt.csv     | 1045 -> 1045 |  213 ->  173
m_all.csv                 | 1045 -> 1045 |  142 ->  119
m_all_OH_rebuilt.csv      | 1045 -> 1045 |  454 ->  427
m_all_FP_rebuilt.csv      | 1045 -> 1045 |  348 ->  283


In [ ]:
Dataset	Variable Selection	Chemical representation	Original	Imputation
A	7-variable selection			358×7	1,045×7
B	7-variable selection		one-hot（OH）	358×137	1,045×318
C	7-variable selection	Fingerprint（FP）		358×91	1,045×173
D	500-sample set			72×24	1,045×119
E	500-sample set		one-hot（OH）	72×48	1,045×427
F	500-sample set	Fingerprint（FP）		72×44	1,045×283
